In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SCHEMA_FILE = PROJECT_ROOT / "tests/fixtures/crypto_schema.json"
RAW_FILE = PROJECT_ROOT / "tests/fixtures/sample_crypto.ndjson"
sys.path.append(str(PROJECT_ROOT))

print("Project Root:", PROJECT_ROOT)
print("Schema_file:", SCHEMA_FILE)
print("Raw_file:", RAW_FILE)

Project Root: /home/diwakar1977/projects/crypto-etl-pipeline
Schema_file: /home/diwakar1977/projects/crypto-etl-pipeline/tests/fixtures/crypto_schema.json
Raw_file: /home/diwakar1977/projects/crypto-etl-pipeline/tests/fixtures/sample_crypto.ndjson


In [4]:
from src.schemas.schema_manager import SchemaManager
from src.utils.spark_session import SparkSessionManager

spark = SparkSessionManager.get_session()

manager = SchemaManager(SCHEMA_FILE)
spark_schema = manager.build_schema()

df = (
    spark.read
    .schema(spark_schema)
    .json(str(RAW_FILE))
)

print("totlal:", df.count())
print("columns:", len(df.columns))

2026-08-06 11:22:16,880 - INFO - spark_session - ============================================================
2026-08-06 11:22:16,883 - INFO - spark_session - Creating Spark Session
2026-08-06 11:22:16,886 - INFO - spark_session - ============================================================
2026-08-06 11:22:16,887 - INFO - spark_session - Running Spark in LOCAL mode.
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/06 11:22:21 WARN Utils: Your hostname, LAPTOP-I2BK1TTC, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/06 11:22:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/diwakar1977/projects/crypto-etl-pipeline/venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/diwakar1977/.ivy2.5.2/cache
The jars for the packages stored in: /home/diwakar1977/.i

totlal: 100
columns: 26


In [5]:
df.show(5,truncate=False)

26/08/06 11:23:30 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----------+------+--------+------------------------------------------------------------------------------------+-------------+-------------+---------------+-----------------------+---------------+--------+--------+----------------+---------------------------+---------------------+--------------------------------+---------------------+--------------------+----------+--------+---------------------+-------------------+---------+---------------------+-------------------+-------------------+----------------------------------------------------------------------------------+
|id         |symbol|name    |image                                                                               |current_price|market_cap   |market_cap_rank|fully_diluted_valuation|total_volume   |high_24h|low_24h |price_change_24h|price_change_percentage_24h|market_cap_change_24h|market_cap_change_percentage_24h|circulating_supply   |total_supply        |max_supply|ath     |ath_change_percentage|ath_date           |at

In [6]:
df.printSchema()

root
 |-- id: string (nullable = true)
 |-- symbol: string (nullable = true)
 |-- name: string (nullable = true)
 |-- image: string (nullable = true)
 |-- current_price: double (nullable = true)
 |-- market_cap: long (nullable = true)
 |-- market_cap_rank: long (nullable = true)
 |-- fully_diluted_valuation: long (nullable = true)
 |-- total_volume: double (nullable = true)
 |-- high_24h: double (nullable = true)
 |-- low_24h: double (nullable = true)
 |-- price_change_24h: double (nullable = true)
 |-- price_change_percentage_24h: double (nullable = true)
 |-- market_cap_change_24h: double (nullable = true)
 |-- market_cap_change_percentage_24h: double (nullable = true)
 |-- circulating_supply: double (nullable = true)
 |-- total_supply: double (nullable = true)
 |-- max_supply: double (nullable = true)
 |-- ath: double (nullable = true)
 |-- ath_change_percentage: double (nullable = true)
 |-- ath_date: timestamp (nullable = true)
 |-- atl: double (nullable = true)
 |-- atl_change_pe

In [7]:
drop_cols = ["roi"]
df = df.drop(*drop_cols)

In [8]:
df.describe().show()

+-------+-----+--------+----------------------+--------------------+-----------------+--------------------+-----------------+-----------------------+-------------------+-----------------+-----------------+------------------+---------------------------+---------------------+--------------------------------+--------------------+--------------------+--------------------+------------------+---------------------+------------------+---------------------+
|summary|   id|  symbol|                  name|               image|    current_price|          market_cap|  market_cap_rank|fully_diluted_valuation|       total_volume|         high_24h|          low_24h|  price_change_24h|price_change_percentage_24h|market_cap_change_24h|market_cap_change_percentage_24h|  circulating_supply|        total_supply|          max_supply|               ath|ath_change_percentage|               atl|atl_change_percentage|
+-------+-----+--------+----------------------+--------------------+-----------------+--------

In [9]:
duplicate_count = (
    df.groupBy("id","symbol")
    .count()
    .filter("count > 1")
    .show()
)

print("Duplicate Records:", duplicate_count)
df = df.dropDuplicates(["id","symbol"])

+---+------+-----+
| id|symbol|count|
+---+------+-----+
+---+------+-----+

Duplicate Records: None


In [10]:
from pyspark.sql.functions import col,when,sum
null_counts = df.select([
    sum(when(col(c).isNull(),1).otherwise(0)).alias(c)
    for c in df.columns
])

null_counts.show()

+---+------+----+-----+-------------+----------+---------------+-----------------------+------------+--------+-------+----------------+---------------------------+---------------------+--------------------------------+------------------+------------+----------+---+---------------------+--------+---+---------------------+--------+------------+
| id|symbol|name|image|current_price|market_cap|market_cap_rank|fully_diluted_valuation|total_volume|high_24h|low_24h|price_change_24h|price_change_percentage_24h|market_cap_change_24h|market_cap_change_percentage_24h|circulating_supply|total_supply|max_supply|ath|ath_change_percentage|ath_date|atl|atl_change_percentage|atl_date|last_updated|
+---+------+----+-----+-------------+----------+---------------+-----------------------+------------+--------+-------+----------------+---------------------------+---------------------+--------------------------------+------------------+------------+----------+---+---------------------+--------+---+----------

In [11]:
from pyspark.sql.functions import col, count, when
from pyspark.sql.types import (
    IntegerType,
    LongType,
    DoubleType,
    FloatType,
    ShortType,
    DecimalType
)

# Automatically detect numeric columns
numeric_columns = [
    field.name
    for field in df.schema.fields
    if isinstance(
        field.dataType,
        (
            IntegerType,
            LongType,
            DoubleType,
            FloatType,
            ShortType,
            DecimalType,
        ),
    )
]

print("Numeric Columns:")
print(numeric_columns)

# Count zero values in every numeric column
df.select([
    count(
        when(col(column) == 0, column)
    ).alias(column)
    for column in numeric_columns
]).show(truncate=False)

Numeric Columns:
['current_price', 'market_cap', 'market_cap_rank', 'fully_diluted_valuation', 'total_volume', 'high_24h', 'low_24h', 'price_change_24h', 'price_change_percentage_24h', 'market_cap_change_24h', 'market_cap_change_percentage_24h', 'circulating_supply', 'total_supply', 'max_supply', 'ath', 'ath_change_percentage', 'atl', 'atl_change_percentage']
+-------------+----------+---------------+-----------------------+------------+--------+-------+----------------+---------------------------+---------------------+--------------------------------+------------------+------------+----------+---+---------------------+---+---------------------+
|current_price|market_cap|market_cap_rank|fully_diluted_valuation|total_volume|high_24h|low_24h|price_change_24h|price_change_percentage_24h|market_cap_change_24h|market_cap_change_percentage_24h|circulating_supply|total_supply|max_supply|ath|ath_change_percentage|atl|atl_change_percentage|
+-------------+----------+---------------+------------

In [12]:
from pyspark.sql.functions import round, col
round_columns = [
    "current_price",
    "high_24h",
    "low_24h",
    "price_change_24h",
    "price_change_percentage_24h",
    "market_cap_change_24h",
    "market_cap_change_percentage_24h",
    "ath",
    "ath_change_percentage",
    "atl",
    "atl_change_percentage"
]

for column in round_columns:
    df = df.withColumn(
        column,
        round(col(column), 2)
    )

In [13]:
from pyspark.sql.functions import current_timestamp
df = df.withColumn(
    "ingest_timestamp",
    current_timestamp()
)

In [14]:
from pyspark.sql.functions import current_date, datediff

df = (
    df
    .withColumn(
        "days_since_ath",
        datediff(current_date(), "ath_date")
    )
    .withColumn(
        "days_since_atl",
        datediff(current_date(), "atl_date")
    )
)

In [15]:
df = df.withColumn(
    "daily_volatility_percentage",
    round(
        ((df.high_24h - df.low_24h) / df.current_price)
        , 2
    )
)

In [16]:
df = df.withColumn(
    "distance_from_ath",
    round(
        ((df.ath - df.current_price) / df.ath)
        ,2
    )
)

In [17]:
df = df.withColumn(
    "distance_from_atl",
    round(
        ((df.atl - df.current_price) / df.atl)
        ,2
    )
)

In [18]:
from pyspark.sql.functions import round,expr
df = df.withColumn(
    "volume_market_cap_ratio",
    round(
        expr("try_divide(total_volume, market_cap)")
        ,4
    )
)

In [19]:
df = df.withColumn(
    "supply_utilization_pct",
    round(
        df.circulating_supply / df.max_supply,
        2
    )
)

In [20]:
df = df.withColumn(
    "price_direction",
    when(df.price_change_24h > 0,"UP")
    .when(df.price_change_24h < 0,"DOWN")
    .otherwise("FLATE")
)

In [21]:
df.show(5,truncate=False)

+--------+------+--------+-----------------------------------------------------------------------------------------------------------+-------------+----------+---------------+-----------------------+------------+--------+-------+----------------+---------------------------+---------------------+--------------------------------+--------------------+-------------------+----------+------+---------------------+-------------------+-----+---------------------+-------------------+-------------------+--------------------------+--------------+--------------+---------------------------+-----------------+-----------------+-----------------------+----------------------+---------------+
|id      |symbol|name    |image                                                                                                      |current_price|market_cap|market_cap_rank|fully_diluted_valuation|total_volume|high_24h|low_24h|price_change_24h|price_change_percentage_24h|market_cap_change_24h|market_cap_change_per